In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

sys.path.append("..")

import numpy as np
import torch

from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.models.light_gcot import LightGCOT
from src.utils.plotting.matplotlib import plot_A_parameters, plot_B_parameters, plot_distributions

In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

In [ ]:
torch.set_default_device(device)

In [ ]:
models = []
n_potentials = 500
m_potentials = 25
epsilon = 0.002
exp_cost = "MLP"
exp_cost_included = True
exp_meta_info = "_c(x,y)=||x+y||^2_rand"
max_steps = 20000

for eps in [epsilon]:
    EXP_NAME = (
        f"LightGCOT_Swiss_Roll_EPSILON_{eps}_MAX_STEPS_{max_steps}_N_{n_potentials}_M_{m_potentials}_with_{exp_cost}_cost_included_{exp_cost_included}"
        + exp_meta_info
    )
    OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)
    D = LightGCOT(
        x_dim=2,
        y_dim=2,
        n_potentials=n_potentials,
        m_potentials=m_potentials,
        epsilon=eps,
        A_diagonal_init=0.1,
        cost_function=exp_cost,
    )
    D.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_{max_steps}.pt")))
    models.append(D)

In [ ]:
X_sampler = StandardNormalSampler(dim=2, device="cuda")
Y_sampler = SwissRollSampler(dim=2, device="cuda")

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Plotting

In [ ]:
starting_points = torch.tensor([[-1.5, 1.5], [0.0, 0.0], [1.5, -1.5]])

In [ ]:
plot_A_parameters(D)

In [ ]:
plot_B_parameters(D, starting_points)

In [ ]:
plot_distributions(D, X_sampler, Y_sampler, starting_points)